# Logistic Regression — UCI Bank Marketing Dataset

Binary classification task: predict whether a customer will subscribe to a term deposit.

**Model:** LogisticRegression (class_weight='balanced')  
**Dataset:** UCI Bank Marketing — 45,211 samples, 16 features  
**Target:** Binary — subscribed (yes/no)  
**Deployment:** Amazon SageMaker (sklearn 1.2-1 container)

In [ ]:
import io
import json
import warnings
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
CACHE = Path('../ml/classification/bank_marketing.csv')
DATASET_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip'

if CACHE.exists():
    df = pd.read_csv(CACHE, sep=';')
    print(f'Loaded from cache: {CACHE}')
else:
    print('Downloading dataset...')
    resp = requests.get(DATASET_URL, timeout=30)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        name = [n for n in z.namelist() if n.endswith('.csv')][0]
        df = pd.read_csv(z.open(name), sep=';')
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False, sep=';')

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Target distribution:')
print(df['y'].value_counts())
print(f'\nClass imbalance: {df["y"].value_counts(normalize=True).round(3).to_dict()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['y'].value_counts().plot(kind='bar', ax=axes[0], color=['#3b82f6','#16a34a'])
axes[0].set_title('Target Class Distribution')
axes[0].set_xlabel('Subscribed')

df.groupby('y')['age'].plot(kind='hist', bins=30, alpha=0.6, ax=axes[1],
                              legend=True, title='Age Distribution by Class')
axes[1].set_xlabel('Age')
plt.tight_layout()
plt.show()

In [ ]:
numeric_features     = ['age','balance','duration','campaign','pdays','previous']
categorical_features = ['job','marital','education','default','housing','loan',
                        'contact','month','poutcome']

df[numeric_features].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, numeric_features):
    for label, color in [('no','#3b82f6'),('yes','#16a34a')]:
        df[df['y']==label][feat].hist(bins=30, alpha=0.6, ax=ax, color=color, label=label)
    ax.set_title(feat)
    ax.legend()
plt.suptitle('Numeric Feature Distributions by Target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Preprocessing + Train/Test Split

In [ ]:
X = df.drop(columns=['y'])
y = (df['y'] == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test  positive rate: {y_test.mean():.3f}')

## 4. Model Training

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        solver='lbfgs',
    )),
])

print('Training Logistic Regression...')
pipeline.fit(X_train, y_train)
print('Done!')

## 5. Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=== Test Set Metrics ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['no','yes']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No','Yes'], yticklabels=['No','Yes'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#3b82f6', linewidth=2, label=f'ROC (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Save Model Artifact

In [ ]:
OUT_DIR = Path('../ml/classification')
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUT_DIR / 'model.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved → {model_path}')

metrics = {
    'model_type': 'LogisticRegression',
    'dataset': 'UCI Bank Marketing',
    'n_train': len(X_train),
    'n_test': len(X_test),
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1': float(f1),
    'roc_auc': float(auc),
    'confusion_matrix': cm.tolist(),
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
}
(OUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Metrics saved → {OUT_DIR / "metrics.json"}')